In [1]:
from pathlib import Path
import sys

import duckdb
import pandas as pd


# ============================================================
# PROJECT CONFIG
# ============================================================

current = Path.cwd().resolve()

PROJECT_ROOT = next(
    p for p in [current, *current.parents]
    if (p / "src" / "data_pipeline" / "config.py").exists()
)

sys.path.insert(
    0,
    str(PROJECT_ROOT / "src" / "data_pipeline")
)

from config import (
    DEVICE_CSV,
    EMAIL_CSV,
    FILE_CSV,
    LOGON_CSV,
    HTTP_CSV,
    INSIDERS_CSV,
)


print("=" * 60)
print("ENVIRONMENT READY")
print("=" * 60)
print(f"Project root : {PROJECT_ROOT}")
print(f"Python       : {sys.version.split()[0]}")
print(f"Pandas       : {pd.__version__}")
print(f"DuckDB       : {duckdb.__version__}")

ENVIRONMENT READY
Project root : C:\Users\HP\Documents\FYP\Data_engineering
Python       : 3.12.7
Pandas       : 2.2.2
DuckDB       : 1.5.5


In [3]:
# ============================================================
# BASIC DATA AUDIT
# ============================================================

files = {
    "device": DEVICE_CSV,
    "email": EMAIL_CSV,
    "file": FILE_CSV,
    "logon": LOGON_CSV,
    "http": HTTP_CSV,
}


def db_path(path):
    return Path(path).resolve().as_posix()


con = duckdb.connect()

print("\n" + "=" * 70)
print("BASIC CERT R4.2 DATA AUDIT")
print("=" * 70)


for name, path in files.items():

    print("\n" + "-" * 70)
    print(f"{name.upper()} DATA")
    print("-" * 70)

    if not Path(path).exists():
        print(f"❌ File not found: {path}")
        continue

    qpath = db_path(path)

    source = f"""
        read_csv_auto(
            '{qpath}',
            types={{'date': 'VARCHAR'}}
        )
    """

    # --------------------------------------------------------
    # Core summary in one scan
    # --------------------------------------------------------

    summary = con.execute(f"""
        SELECT
            COUNT(*) AS rows,
            COUNT(DISTINCT "user") AS unique_users,

            MIN(
                try_strptime(
                    date,
                    '%m/%d/%Y %H:%M:%S'
                )
            ) AS first_date,

            MAX(
                try_strptime(
                    date,
                    '%m/%d/%Y %H:%M:%S'
                )
            ) AS last_date,

            SUM(
                CASE
                    WHEN date IS NOT NULL
                     AND try_strptime(
                         date,
                         '%m/%d/%Y %H:%M:%S'
                     ) IS NULL
                    THEN 1
                    ELSE 0
                END
            ) AS invalid_dates

        FROM {source}
    """).fetchone()

    print(f"Rows            : {summary[0]:,}")
    print(f"Unique users    : {summary[1]:,}")
    print(f"First date      : {summary[2]}")
    print(f"Last date       : {summary[3]}")
    print(f"Invalid dates   : {summary[4]}")

    # --------------------------------------------------------
    # Sample rows
    # --------------------------------------------------------

    sample = con.execute(f"""
        SELECT *
        FROM {source}
        LIMIT 3
    """).df()

    print("\nSample:")
    display(sample)

    # --------------------------------------------------------
    # Activity distribution
    # --------------------------------------------------------

    if name in {"device", "logon"}:

        activity = con.execute(f"""
            SELECT
                activity,
                COUNT(*) AS count
            FROM {source}
            GROUP BY activity
            ORDER BY count DESC
        """).df()

        print("\nActivity distribution:")
        display(activity)


# ============================================================
# INSIDER GROUND TRUTH
# ============================================================

print("\n" + "=" * 70)
print("CERT R4.2 INSIDER GROUND TRUTH")
print("=" * 70)

if INSIDERS_CSV.exists():

    insiders = pd.read_csv(
        INSIDERS_CSV,
        dtype=str,
    )

    insiders["dataset"] = pd.to_numeric(
        insiders["dataset"],
        errors="coerce",
    )

    r42 = insiders[
        insiders["dataset"] == 4.2
    ].copy()

    print(f"Total incidents         : {len(r42)}")
    print(f"Unique malicious users  : {r42['user'].nunique()}")

    print("\nScenario distribution:")
    display(
        r42["scenario"]
        .value_counts()
        .sort_index()
        .rename_axis("scenario")
        .reset_index(name="count")
    )

    print("\nMalicious users:")
    print(
        ", ".join(
            sorted(
                r42["user"]
                .dropna()
                .unique()
            )
        )
    )

    print("\nGround-truth sample:")
    display(r42.head())

else:
    print(f"❌ insiders.csv not found: {INSIDERS_CSV}")


con.close()

print("\n" + "=" * 70)
print("✅ DATA AUDIT COMPLETE")
print("=" * 70)


BASIC CERT R4.2 DATA AUDIT

----------------------------------------------------------------------
DEVICE DATA
----------------------------------------------------------------------
Rows            : 405,380
Unique users    : 265
First date      : 2010-01-02 07:21:06
Last date       : 2011-05-16 23:22:34
Invalid dates   : 0

Sample:


,id,date,user,pc,activity
0,{J1S3-L9UU75BQ-7790ATPL},01/02/2010 07:21:06,MOH0273,PC-6699,Connect
1,{N7B5-Y7BB27SI-2946PUJK},01/02/2010 07:37:41,MOH0273,PC-6699,Disconnect
2,{U1V9-Z7XT67KV-5649MYHI},01/02/2010 07:59:11,HPH0075,PC-2417,Connect



Activity distribution:


,activity,count
0,Connect,203339
1,Disconnect,202041



----------------------------------------------------------------------
EMAIL DATA
----------------------------------------------------------------------
Rows            : 2,629,979
Unique users    : 1,000
First date      : 2010-01-02 07:11:45
Last date       : 2011-05-16 21:16:26
Invalid dates   : 0

Sample:


,id,date,user,pc,to,cc,bcc,from,size,attachments,content
0,{R3I7-S4TX96FG-8219JWFF},01/02/2010 07:11:45,LAP0338,PC-5758,Dean.Flynn.Hines@dtaa.com;Wade_Harrison@lockhe...,Nathaniel.Hunter.Heath@dtaa.com,None,Lynn.Adena.Pratt@dtaa.com,25830,0,middle f2 systems 4 july techniques powerful d...
1,{R0R9-E4GL59IK-2907OSWJ},01/02/2010 07:12:16,MOH0273,PC-6699,Odonnell-Gage@bellsouth.net,None,None,MOH68@optonline.net,29942,0,the breaking called allied reservations former...
2,{G2B2-A8XY58CP-2847ZJZL},01/02/2010 07:13:00,LAP0338,PC-5758,Penelope_Colon@netzero.com,None,None,Lynn_A_Pratt@earthlink.net,28780,0,slowly this uncinus winter beneath addition ex...


... (11 outputs left out)

,scenario,count
0,1,30
1,2,30
2,3,10



Malicious users:
AAF0535, AAM0658, ABC0174, AJR0932, AKR0057, BBS0039, BDV0168, BIH0745, BLS0678, BSS0369, BTL0226, CAH0936, CCA0046, CCL0068, CEJ0109, CQW0652, CSC0217, DCH0843, DIB0285, DRR0162, EDB0714, EGD0132, EHB0824, EHD0584, FMG0527, FSC0601, FTM0406, GHL0460, GTD0219, HBO0413, HJB0742, HXL0968, IJM0776, IKR0401, IUB0565, JGT0221, JJM0203, JLM0364, JMB0308, JRG0207, JTM0223, KLH0596, KPC0073, KRL0501, LCC0819, LJR0523, LQC0479, MAR0955, MAS0025, MCF0600, MDH0580, MOS0047, MPM0220, MSO0222, MYD0978, NWT0098, PNL0301, PPF0435, PSF0133, RAB0589, RAR0725, RGG0064, RHL0992, RKD0604, RMW0542, TAP0551, TNM0961, VSS0154, WDD0366, XHW0498

Ground-truth sample:


,dataset,scenario,details,user,start,end
8,4.2,1,r4.2-1-AAM0658.csv,AAM0658,10/23/2010 01:34:19,10/29/2010 05:23:28
9,4.2,1,r4.2-1-AJR0932.csv,AJR0932,09/10/2010 19:12:01,09/18/2010 02:02:51
10,4.2,1,r4.2-1-BDV0168.csv,BDV0168,07/30/2010 19:56:44,08/10/2010 05:16:41
11,4.2,1,r4.2-1-BIH0745.csv,BIH0745,07/13/2010 20:15:23,07/13/2010 21:20:44
12,4.2,1,r4.2-1-BLS0678.csv,BLS0678,09/21/2010 01:16:22,09/30/2010 04:48:19



✅ DATA AUDIT COMPLETE
